# Feature Engineering & Derived Business Columns

This notebook demonstrates feature engineering techniques for converting raw transaction data into high-value business metrics.

### Tasks Covered:
1. **Ratio Features**: Normalizing raw counts by time and volume (`transactions_per_month`, `avg_spend_per_transaction`, `lifetime_value_per_month`).
2. **Equal-Width Binning (`pd.cut`)**: Segmenting activity rates into discrete business tiers (`low`, `medium`, `high`).
3. **Quantile Binning (`pd.qcut`)**: Dividing monetary spend into balanced quartiles (`Q1`, `Q2`, `Q3`, `Q4`).
4. **Composite Scoring**: Constructing an RFM (Recency, Frequency, Monetary) composite score.
5. **Feature Validation**: Ensuring valid ranges and zero NaN values.

In [ ]:
import pandas as pd
import numpy as np
import os

# Load raw dataset
df = pd.read_csv('../data/raw/feature_engineering_data.csv')
if 'purchase_count' not in df.columns:
    df['purchase_count'] = df['total_transactions']
df.head()

## Task 1: Compute Ratio Features

Compute rate and intensity metrics to contextualize raw counts.

In [ ]:
df['transactions_per_month'] = df['total_transactions'] / (df['days_as_customer'] / 30)
df['avg_spend_per_transaction'] = df['total_spent'] / df['total_transactions']
df['lifetime_value_per_month'] = df['total_spent'] / (df['days_as_customer'] / 30)

print(df[['transactions_per_month', 'avg_spend_per_transaction', 'lifetime_value_per_month']].describe())

## Task 2: Binning with Equal-Width Bins

Use `pd.cut` with fixed boundaries to define engagement tiers.

In [ ]:
df['engagement_tier'] = pd.cut(
    df['transactions_per_month'],
    bins=[0, 2, 10, float('inf')],
    labels=['low', 'medium', 'high']
)

print(df['engagement_tier'].value_counts())

## Task 3: Binning with Quantiles

Use `pd.qcut` to divide customers into equal-sized spend quartiles.

In [ ]:
df['spend_quartile'] = pd.qcut(
    df['total_spent'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

print(df['spend_quartile'].value_counts())

## Task 4: Composite Score

Build a composite RFM score from Recency, Frequency, and Monetary ranks.

In [ ]:
df['recency_score'] = pd.qcut(df['days_since_last_purchase'], q=5, labels=[5,4,3,2,1])
df['frequency_score'] = pd.qcut(df['purchase_count'].rank(method='first'), q=5, labels=[1,2,3,4,5])
df['monetary_score'] = pd.qcut(df['total_spent'].rank(method='first'), q=5, labels=[1,2,3,4,5])

df['rfm_score'] = (df['recency_score'].astype(int) + 
                   df['frequency_score'].astype(int) + 
                   df['monetary_score'].astype(int))

print(f"RFM score range: {df['rfm_score'].min()}-{df['rfm_score'].max()}")

## Task 5: Feature Validation

Validate distribution, ranges, and ensure zero NaNs.

In [ ]:
# Check ranges are sensible
print(f"Engagement tier distribution:\n{df['engagement_tier'].value_counts()}")
print(f"RFM score range: {df['rfm_score'].min()}-{df['rfm_score'].max()}")

# Ensure no NaNs introduced
print(f"Missing values:\n{df[['engagement_tier', 'spend_quartile', 'rfm_score']].isna().sum()}")

# Export processed dataset
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/engineered_customer_features.csv', index=False)